In [4]:
# 1.加载环境变量
from langchain.tools import tool
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，描述工具

# 描述方式一：tool装饰器的属性来描述
@tool("square_root", description="计算数字x的平方根")
def tool1(x: float) -> float:
    return x ** 0.5

# 描述方式二：用文档注释来描述
# @tool
# def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
#     """
#     获取当前天气和可选的天气预报。
#     Args:
#         location: 城市名称或坐标
#         units: 温度单位，可选值['celsius', 'fahrenheit']
#         include_forecast: 包括天气预报吗
#     """
#     temp = 22 if units == "celsius" else 72
#     result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
#     if include_forecast:
#         result += "\nNext 5 days: Sunny"
#     return result

# 描述方式三：用BaseModel描述(了解）
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool
# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="城市名称或坐标")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="温度单位偏好"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包括5日预报"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


tools = [tool1, get_weather]

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.invoke调用Agent， 传入的Dict，里面有messages
response = agent.invoke(
    {"messages": [HumanMessage("杭州接下来几天的天气如何")]}
)
print(response)

{'messages': [HumanMessage(content='杭州接下来几天的天气如何', additional_kwargs={}, response_metadata={}, id='0907aea7-7247-4cf8-8746-e3b79f0299c8'), AIMessage(content='我来帮您查询杭州的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 398, 'total_tokens': 470, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 398}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'e593c358-9524-46db-be1f-9dab82a8b45f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff00d-d861-7342-89a6-76643308eb13-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '杭州', 'include_forecast': True}, 'id': 'call_00_zxSoFRRgkiuJ0C0ov6hL4437', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token